# **[_(Bonus) - Data Ingestion with MERGE INTO_](url)**

MERGE INTO in Databricks is a powerfull toll for data ingestion, especially for data ingestion. It enables efficient, atomic, scalable upsert and delete operations. This command is useful when you have an existing Delta table and you wish to combine incomming data.

In [0]:
%sql
USE `pysaprk_demo`.`merge_into_demo`;

In [0]:
data = [
    (1, 'Samarth', 'samarth@example.com', '2024-01-05', 'current'),
    (2, 'Priya', 'priya@example.com', '2024-01-05', 'current'),
    (3, 'Ravi', 'ravi@example.com', '2024-01-05', 'current'),
    (4, 'Mark', 'mark@example.com', '2024-01-05', 'current'),
    (5, 'zebi', 'zebi@example.com', '2024-01-05', 'current')
]

from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType

schema = StructType([
    StructField('id', IntegerType(), True),
    StructField('name', StringType(), True),
    StructField('email', StringType(), True),
    StructField('sign_up_date', StringType(), True),
    StructField('status', StringType(), True)   
])


df = spark.createDataFrame(data, schema)

df.write.mode("overwrite").saveAsTable("tb_main_users_target")

df.show()

df.printSchema()

In [0]:
%sql
SELECT *
FROM tb_main_users_target
;

In [0]:
data = [
    (1, 'Samarth', 'samarth@example.com', '2024-01-05', 'delete'),
    (2, 'Priya', 'priya@newemail.com', '2024-01-05', 'update'),
    (6, 'Owen', 'owen@example.com', '2024-01-05', 'new'),
    (7, 'Eva', 'eva@example.com', '2024-01-05', 'new')
]

from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType

schema = StructType([
    StructField('id', IntegerType(), True),
    StructField('name', StringType(), True),
    StructField('email', StringType(), True),
    StructField('sign_up_date', StringType(), True),
    StructField('status', StringType(), True)   
])


df = spark.createDataFrame(data, schema)

df.write.mode("overwrite").saveAsTable("tb_updated_users_source")

df.show()

df.printSchema()

In [0]:
%sql 
MERGE INTO tb_main_users_target as tar_df
USING tb_updated_users_source as new_df
ON tar_df.id = new_df.id
WHEN MATCHED AND new_df.status = 'update' THEN
UPDATE SET *
WHEN MATCHED AND new_df.status = 'delete' THEN
DELETE
WHEN NOT MATCHED AND new_df.status = 'new' THEN 
  INSERT (id, name, email, sign_up_date, status) 
  VALUES (new_df.id, new_df.name, new_df.email, new_df.sign_up_date, new_df.status)
;

In [0]:
%sql
SELECT *
FROM tb_main_users_target
;

In [0]:
data = [
    (8, 'Kristina', 'kristina@example.com', '2024-01-05', 'new', 'USA'),
    (9, 'John', 'john@example.com', '2024-01-05', 'new', 'USA'),
    (10, 'Jane', 'jane@example.com', '2024-01-05', 'new', 'Canada'),
    (11, 'Alice', 'alice@example.com', '2024-01-05', 'new', 'Canada'),
    (12, 'Mohammed', 'mohammed@example.com', '2024-01-05', 'new', 'Pakistan'),
    (13, 'Bob', 'bob@example.com', '2024-01-05', 'new', 'Canada')
]

schema = StructType([
    StructField('id', IntegerType(), True),
    StructField('name', StringType(), True),
    StructField('email', StringType(), True),
    StructField('sign_up_date', StringType(), True),
    StructField('status', StringType(), True),
    StructField('country', StringType(), True)
])

df = spark.createDataFrame(data, schema)

df.write.mode("overwrite").saveAsTable("tb_new_users_source")

df.show()

df.printSchema()

In [0]:
MERGE INTO tb_main_users_target as tar_df
USING tb_updated_users_source as src_df
ON tar_df.id = src_df.id
WHEN MATCHED AND src_df.status = 'update' THEN 
  UPDATE SET 
    tar_df.status = src_df.status
    ,tar_df.email = src_df.email
    ,tar_df.sign_up_date = src_df.sign_up_date
    ,tar_df.name = src_df.name
WHEN MATCHED and src_df.status = 'delete' THEN 
    DELETE
WHEN NOT MATCHED and src_df.status = 'new' THEN 
    INSERT (id, name, email, sign_up_date, status) 
    VALUES (src_df.id, src_df.name, src_df.email, src_df.sign_up_date, src_df.status)
;

In [0]:
%sql
MERGE WITH SCHEMA EVOLUTION INTO tb_main_users_target as tar_df
USING tb_new_users_source as src_df
ON tar_df.id = src_df.id
WHEN MATCHED AND src_df.status = 'update' THEN 
  UPDATE SET *
WHEN MATCHED AND src_df.status = 'delete' THEN 
    DELETE
WHEN NOT MATCHED AND src_df.status = 'new' THEN 
    INSERT *
;

In [0]:
%sql
SELECT *
FROM tb_main_users_target
;